# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Mar 3rd at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 1)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions


## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 
>
> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.
> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

## 1.
In Centola's exeriment, the dataset was tailor-made and split into a random and clustered network in accordance with their research question which made it easier to answer. However, they had to spend time and resources to construct the dataset. Furthermore, the study participants did not know that they were participating in research which raises an ethical dilemma.

In Nicolaides' study, they collected the dataset from a fitness app, saving them time and resources. A drawback that emerged from using ready-made data, however, is that correlation does not equal causation. In order to rule out homophily, for instance, the study had to make use of instrumental variable design to control for confounding variables.

**Poul's suggestion:**

*Centola's study uses custom-made data from a controlled online experiment. The main advantage is strong internal validity — because participants were randomly assigned to different network structures, the study can establish causation between network topology and behavior spread. Centola also had complete data and full control over variables, eliminating algorithmic confounding. On the downside, the sample was small (~1,500 people), the network was artificial, and participants knew they were in an experiment, introducing reactivity (Salganik, 2.3.3) and limiting external validity.*

*Nicolaides's study relies on ready-made fitness app data. This gives massive scale, always-on longitudinal data, and nonreactive measurement of real behavior at low cost. The trade-offs are significant though: without random assignment, only correlations can be shown. The data is also incomplete (no info on motivation or health), potentially algorithmically confounded by app notifications, and nonrepresentative since app users skew wealthier and younger.*

## 2.
The differences in data collection affect how we interpret the results. In Centola’s custom-made experiment, the controlled design and random assignment allow for stronger causal claims, since alternative explanations (e.g., homophily or confounding factors) are minimized by design. However, the artificial environment might not fully generalize to real-world networks.

In contrast, Nicolaides’ ready-made data come from a natural setting, making the results more reflective of real behavior. Conversely, causal inference is weaker because the data was not generated through random assignment. Even with techniques like instrumental variables, unobserved confounding might always remain a small possibility. Therefore, Centola’s results are more convincing in terms of causality, while Nicolaides’ findings may be stronger in terms of real-world relevance but more cautious in causal interpretation.

**Poul's suggestion:**

*These differences matter a lot for what we can actually conclude. Centola can confidently say that network structure caused differences in behavior spread, since he controlled for everything else. But because the setting was artificial and participants knew about the experiment, it is hard to know whether these dynamics would play out in real social networks. The findings are internally solid but may not generalize.*

*Nicolaides shows patterns of exercise contagion among real people at large scale, which feels more applicable to the real world. But there are many possible confounders — friends might run more at the same time due to shared weather or app-driven challenges, not genuine social influence. The missing variables make it impossible to rule these out. So while the patterns are compelling, they should be interpreted with caution since the study cannot definitively establish causation.*

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025. This will not only later on help you understand the landscape of Computational Social Science research but also develop practical skills in data collection and analysis. 
>
> Please read the text of the whole exercise before starting to work on it. 
>
> **Steps**
> 
> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution
> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.
>
>    
> **Handling Challenges**
> 
> I expect that, while working on the steps above, you will encounter several obstacles. As you complete this exercise, you are expected to:     
>
>    - Identify problems that arise.      
>    - Improve your solutions to address such problems, making reasonable decisions when data is incomplete or ambiguous.       
>
> **Reflection Questions**
>  Answer the following questions __(max 150 words for each question)__: 
>
>    - Which challenges did you encounter? How did you address them?
>    - Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data? 
>      


In [1]:
import pandas as pd
import requests
import time
import unicodedata
import re

In [3]:
# read the file
names_df = pd.read_csv("cleaned_names.csv")

# read name column
canonical_names = names_df["name"].dropna().astype(str).tolist()

print("Loaded names:", len(canonical_names))
print(canonical_names[:10])

Loaded names: 1550
['Aakriti Kumar', 'Aaron Clauset', 'Aaron D Nichols', 'Aaron Reeves', 'Aaron Schein', 'Abdulaziz Alhumaidy', 'Abdullah Almaatouq', 'Abhisek Dash', 'Abraham Israeli', 'Achim Edelmann']


Compare similar names:

In [4]:
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.replace(".", "").replace("’", "'").replace("´", "'")
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def candidate_best_name_match_score(query_name: str, cand: dict) -> int:
    q = norm(query_name)

    names_to_compare = [cand.get("display_name", "")]
    names_to_compare += cand.get("display_name_alternatives") or []

    # exact match on any representation wins
    for n in names_to_compare:
        if norm(n) == q:
            return 100

    # score = how many tokens overlap
    q_tokens = set(q.split())
    best_overlap = 0
    for n in names_to_compare:
        t = set(norm(n).split())
        best_overlap = max(best_overlap, len(q_tokens & t))

    return best_overlap


Fetch info

In [7]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (IC2S2 name lookup)"})

PER_PAGE = 5
OPENALEX_API_KEY = "NkafC7tV5c7anrujEZT7xI".strip()

def fetch_author_row(query_name: str) -> dict:
    url = "https://api.openalex.org/authors"
    params = {
        "search": query_name,
        "per-page": PER_PAGE,
        "select": ",".join([
            "id",
            "display_name",
            "display_name_alternatives",
            "works_api_url",
            "works_count",
            "summary_stats",
            "last_known_institutions"
        ]),
        "api_key": OPENALEX_API_KEY
    }

    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    results = data.get("results", [])

    if not results:
        return {
            "query_name": query_name,
            "id": None,
            "display_name": None,
            "works_api_url": None,
            "h_index": None,
            "works_count": None,
            "country_code": None,
            "matched": False
        }

    best = max(results, key=lambda c: candidate_best_name_match_score(query_name, c))

    # h-index lives in summary_stats
    summary = best.get("summary_stats") or {}
    h_index = summary.get("h_index")

    # last_known_institutions is a LIST; take the first one (most recent)
    insts = best.get("last_known_institutions") or []
    country_code = insts[0].get("country_code") if insts else None

    return {
        "query_name": query_name,
        "id": best.get("id"),
        "display_name": best.get("display_name"),
        "works_api_url": best.get("works_api_url"),
        "h_index": h_index,
        "works_count": best.get("works_count"),
        "country_code": country_code,
        "matched": True
    }

Run code:

In [ ]:
SLEEP = 0.12
rows = []

for i, name in enumerate(canonical_names, 1):
    try:
        rows.append(fetch_author_row(name))
    except Exception as e:
        rows.append({
            "query_name": name,
            "id": None,
            "display_name": None,
            "works_api_url": None,
            "h_index": None,
            "works_count": None,
            "country_code": None,
            "matched": False,
            "error": str(e)
        })
    # Print progress
    if i % 25 == 0:
        print(f"Processed {i}/{len(canonical_names)}")
    time.sleep(SLEEP)

authors_df = pd.DataFrame(rows)
authors_df.to_csv("openalex_authors_from_cleaned_names.csv", index=False) # Save list
authors_df.head()

Processed 25/1550
Processed 50/1550
Processed 75/1550
Processed 100/1550
Processed 125/1550
Processed 150/1550
Processed 175/1550
Processed 200/1550
Processed 225/1550
Processed 250/1550
Processed 275/1550
Processed 300/1550
Processed 325/1550
Processed 350/1550
Processed 375/1550
Processed 400/1550
Processed 425/1550
Processed 450/1550
Processed 475/1550
Processed 500/1550
Processed 525/1550
Processed 550/1550
Processed 575/1550
Processed 600/1550
Processed 625/1550
Processed 650/1550
Processed 675/1550
Processed 700/1550
Processed 725/1550
Processed 750/1550
Processed 775/1550
Processed 800/1550
Processed 825/1550
Processed 850/1550
Processed 875/1550
Processed 900/1550
Processed 925/1550
Processed 950/1550
Processed 975/1550
Processed 1000/1550
Processed 1025/1550
Processed 1050/1550
Processed 1075/1550
Processed 1100/1550
Processed 1125/1550
Processed 1150/1550
Processed 1175/1550
Processed 1200/1550
Processed 1225/1550
Processed 1250/1550
Processed 1275/1550
Processed 1300/1550
Pr

,query_name,id,display_name,works_api_url,h_index,works_count,country_code,matched,error
0,Aakriti Kumar,https://openalex.org/A5040234818,Aakriti Kumar,https://api.openalex.org/works?filter=author.i...,9.0,20.0,US,True,NaN
1,Aaron Clauset,https://openalex.org/A5014647140,Aaron Clauset,https://api.openalex.org/works?filter=author.i...,47.0,267.0,US,True,NaN
2,Aaron D Nichols,https://openalex.org/A5089395967,Aaron Nichols,https://api.openalex.org/works?filter=author.i...,3.0,10.0,US,True,NaN
3,Aaron Reeves,https://openalex.org/A5039995516,Aaron Reeves,https://api.openalex.org/works?filter=author.i...,48.0,254.0,GB,True,NaN
4,Aaron Schein,https://openalex.org/A5021334672,Joshua Maynez,https://api.openalex.org/works?filter=author.i...,12.0,51.0,NaN,True,NaN


## Reflection Questions
Some researcher names appeared to be similar or slightly different compared to names on OpenAlex. To account for this, a norm function for comparison was implemented. It compared names based on token overlap. 

While fetching the data, we would sometimes get 429 Too Many Requests. Therefore, we set a sleep timer to 0.12 to prevent this. 


## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*. **Before you start, please ensure you read through the entire exercise.**
>
> 
> **Steps:**
>  
> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.
> 
>     **Important Note on Paging:** By default, the OpenAlex API limits responses to 25 works per request. For more efficient data retrieval, I suggest to adjust this limit to 200 works per request. Even with this adjustment, you will need to implement pagination to access all available works for a given query. This ensures you can systematically retrieve the complete set of works beyond the initial 200. Find guidance on implementing pagination [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging#cursor-paging).
>
> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.
>  
>
> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.
>
>
>   
> For reference, employing these strategies allowed me to fetch the data in about 30 seconds using 5 cores on my laptop. I obtained a dataset of approximately 25 MB (including both the *IC2S2 abstracts* and *IC2S2 papers* files).
> 
>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__: 
> 
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     
> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?    
> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

In [9]:
import json
import numpy as np

In [10]:
api_key = "NkafC7tV5c7anrujEZT7xI"
BASE_URL = "https://api.openalex.org/works"

In [11]:
authors_df = pd.read_csv("openalex_authors_from_cleaned_names.csv")

# keep only authors with valid OpenAlex IDs
authors_df = authors_df.dropna(subset=["id"])

# filter work count between 5 and 5000
authors_df = authors_df[
    (authors_df["works_count"] >= 5) &
    (authors_df["works_count"] <= 5000)
]

author_ids = authors_df["id"].tolist()

print("Authors after filter:", len(author_ids))

Authors after filter: 838


In [12]:
def chunks(lst, n=25):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

author_chunks = list(chunks(author_ids, 25))
print("Number of author chunks:", len(author_chunks))


Number of author chunks: 34


In [13]:
PER_PAGE = 200
SLEEP = 0.2

def fetch_works_for_author_chunk(author_chunk):

    author_filter = "|".join(author_chunk)

    # OR groups must be grouped separately
    css_concepts = "C144024400|C15744967|C162324750|C17744445" # computational social science
    quant_concepts = "C33923547|C121332964|C41008148" # quantitative disciplines

    params = {
        "filter": (
            f"author.id:{author_filter},"
            f"cited_by_count:>10,"
            f"authors_count:<10,"
            f"concepts.id:{css_concepts},"
            f"concepts.id:{quant_concepts}"
        ),
        "per-page": PER_PAGE,
        "cursor": "*",
        "select": ",".join([
            "id",
            "publication_year",
            "cited_by_count",
            "authorships",
            "title",
            "abstract_inverted_index"
        ]),
        "api_key": api_key
    }

    works = []

    while True:
        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        works.extend(data["results"])

        cursor = data["meta"]["next_cursor"]
        if not cursor:
            break

        params["cursor"] = cursor
        time.sleep(SLEEP)

    return works


In [20]:
all_works = []

for chunk in author_chunks:
    works = fetch_works_for_author_chunk(chunk)
    all_works.extend(works)

print("Total works collected:", len(all_works))

Total works collected: 10690


In [21]:
with open("all_works_raw.json", "w") as f:
    json.dump(all_works, f)

print("Saved raw works:", len(all_works))

Saved raw works: 10690


In [22]:
papers = []
abstracts = []

for w in all_works:
    author_ids = [a["author"]["id"] for a in w.get("authorships", [])]

    papers.append({
        "id": w.get("id"),
        "publication_year": w.get("publication_year"),
        "cited_by_count": w.get("cited_by_count"),
        "author_ids": author_ids
    })

    abstracts.append({
        "id": w.get("id"),
        "title": w.get("title"),
        "abstract_inverted_index": w.get("abstract_inverted_index")
    })

D2 = pd.DataFrame(papers).drop_duplicates(subset=["id"])
D3 = pd.DataFrame(abstracts).drop_duplicates(subset=["id"])

D2["author_ids"] = D2["author_ids"].apply(
    lambda x: json.dumps(x) if isinstance(x, list) else json.dumps([])
)

# Save
D2.to_parquet("D2_IC2S2_papers.parquet", index=False)
D3.to_parquet("D3_IC2S2_abstracts.parquet", index=False)

print("D2 size:", len(D2))
print("D3 size:", len(D3))


D2 size: 9870
D3 size: 9870
